In [ ]:
!pip install -U datasets
!pip install langchain langchain_huggingface langchain_community faiss-cpu
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
# Install required packages for different LLM providers
!pip install python-dotenv langchain-google-genai langchain-openai

In [ ]:
# change API token
!huggingface-cli login --token "api_token"

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `nothing in here` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `nothing in here`


In [ ]:
from datasets import load_dataset
import pandas as pd
import os
import torch

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain.retrievers.document_transformers import EmbeddingsRedundantFilter
from langchain_core.retrievers import BaseRetriever

In [ ]:
# Cấu hình API keys từ file .env
from dotenv import load_dotenv

# Load API keys từ file .env
load_dotenv()

# Kiểm tra các API keys đã được tải
print("API keys đã được tải:")
print(f"GOOGLE_API_KEY: {'Đã tải' if os.getenv('GOOGLE_API_KEY') else 'Không tìm thấy'}")
print(f"OPENAI_API_KEY: {'Đã tải' if os.getenv('OPENAI_API_KEY') else 'Không tìm thấy'}")
print(f"HUGGINGFACEHUB_API_TOKEN: {'Đã tải' if os.getenv('HUGGINGFACEHUB_API_TOKEN') else 'Không tìm thấy'}")

In [ ]:
# HuggingFace imports
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

data = load_dataset("harouzie/vi_question_generation", split="train")[:1000]
df = pd.DataFrame(data)
df.to_csv("vi_question_generation.csv", index=True)

df

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/828 [00:00<?, ?B/s]

(…)-00000-of-00001-8a18d84f08d191f2.parquet:   0%|          | 0.00/114M [00:00<?, ?B/s]

(…)-00000-of-00001-7d2ec83abad3f49b.parquet:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

(…)-00000-of-00001-a72038a2db09f9b7.parquet:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/174499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21813 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/21812 [00:00<?, ? examples/s]

,context,question,answers,id
0,Vô số người bản địa đã chiếm Alaska trong hàng...,Dịch bệnh nào dẫn đến nhiều cái chết giữa nhữn...,"{'text': array(['bệnh đậu mùa'], dtype=object)...",572866543acd2414000df995
1,Những mô tả văn hóa của chó trong nghệ thuật k...,"Hàng ngàn năm trước, những chú chó được mô tả ...","{'answer_start': array([109]), 'text': array([...",56d63d821c85041400947055
2,Những hạn chế này đã gây ra những vấn đề. Ví d...,Mặc dù lịch trình trong Israel trở nên dễ đoán...,"{'answer_start': array([778]), 'text': array([...",56e7a23000c9c71400d77447
3,Một kỹ thuật văn học hoặc thiết bị văn học có ...,Theo kiểu viết nào thì một cấu trúc câu chuyện...,"{'answer_start': array([568]), 'text': array([...",5726ef205951b619008f82c1
4,"Tốc độ trung bình rất khác nhau, nhưng thường ...",Tại sao một số sông băng bị đình trệ ở Alaska?,{'text': array(['cây có thể tự thiết lập trên ...,572702ec708984140094d874
...,...,...,...,...
995,Sự thất bại nhanh chóng và quyết định của quân...,Sự thất bại của quân đội Ả Rập ở bàn tay của q...,"{'answer_start': array([12, 12]), 'text': arra...",573005b9947a6a140053cf6a
996,"Vào tháng 10 năm 2010, tạp chí khoa học truy c...",Giấy tờ mầm bệnh yêu cầu gì?,{'text': array(['chứng minh rõ ràng rằng Y. pe...,57264e2f708984140094c1e5
997,Safari và Di động Safari cũng luôn được bao gồ...,Trình duyệt nào được tự động bao gồm với OS X?,"{'answer_start': array([0]), 'text': array(['S...",56e0cf39231d4119001ac3cf
998,Ngôn ngữ học lịch sử nổi lên như một lĩnh vực ...,Chủ đề mới nào xuất hiện vào cuối thế kỷ 18?,"{'text': array(['Ngôn ngữ học lịch sử'], dtype...",57282fda2ca10214002da01e


In [ ]:
# Load and prepare data
df = pd.read_csv("vi_question_generation.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nSample data:")
print(df.head())

Dataset shape: (1000, 5)
Columns: ['Unnamed: 0', 'context', 'question', 'answers', 'id']

Sample data:
   Unnamed: 0                                            context  \
0           0  Vô số người bản địa đã chiếm Alaska trong hàng...   
1           1  Những mô tả văn hóa của chó trong nghệ thuật k...   
2           2  Những hạn chế này đã gây ra những vấn đề. Ví d...   
3           3  Một kỹ thuật văn học hoặc thiết bị văn học có ...   
4           4  Tốc độ trung bình rất khác nhau, nhưng thường ...   

                                            question  \
0  Dịch bệnh nào dẫn đến nhiều cái chết giữa nhữn...   
1  Hàng ngàn năm trước, những chú chó được mô tả ...   
2  Mặc dù lịch trình trong Israel trở nên dễ đoán...   
3  Theo kiểu viết nào thì một cấu trúc câu chuyện...   
4     Tại sao một số sông băng bị đình trệ ở Alaska?   

                                             answers                        id  
0  {'text': array(['bệnh đậu mùa'], dtype=object)...  572866543acd2414

In [ ]:
# Prepare documents for RAG
def prepare_documents(dataframe):
    """Convert dataframe to LangChain documents"""
    documents = []

    # Extract context column if it exists
    if 'context' in dataframe.columns:
        for i, row in dataframe.iterrows():
            if pd.notna(row['context']):
                # Create LangChain document with metadata
                doc = Document(
                    page_content=row['context'],
                    metadata={"source": f"doc_{i}"}
                )
                documents.append(doc)
    else:
        print("No 'context' column found in the dataframe")

    return documents

# Create documents
documents = prepare_documents(df)
print(f"Created {len(documents)} documents")

Created 1000 documents


In [ ]:
# Create text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=100,
    length_function=len
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 1000 chunks


In [ ]:
# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

# Create vector store
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store created successfully!")

Vector store created successfully!


In [ ]:
# Save the vector store for future use
save_path = "faiss_index"
vectorstore.save_local(save_path)
print(f"Vector store saved to {save_path}")

Vector store saved to faiss_index


In [ ]:
# Create simple retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Query Expansion và Reranking

Phần này trình bày các phương pháp nâng cao hiệu suất RAG bằng kỹ thuật mở rộng truy vấn (Query Expansion) và xếp hạng lại kết quả (Reranking).

In [ ]:
# Query Expansion Module
class QueryExpansion:
    """Mở rộng câu truy vấn để tăng khả năng tìm kiếm kết quả liên quan"""
    def __init__(self, model_name=None):
        self.model_name = model_name
        # Mô hình có thể là HF hoặc API-based model
        # Hiện tại để trống, sẽ triển khai trong tương lai
        pass

    def expand_query(self, query):
        """Mở rộng câu truy vấn thành nhiều biến thể"""
        # Trả về câu query gốc khi chưa triển khai model mở rộng
        expanded_queries = [query]
        # TODO: Triển khai mở rộng truy vấn ở đây
        # VD: Thêm các biến thể của câu hỏi, sử dụng LLM để tạo queries tương tự
        return expanded_queries

# Khởi tạo module query expansion
query_expander = QueryExpansion()

In [ ]:
# Reranker Module
class Reranker:
    """Xếp hạng lại kết quả tìm kiếm dựa trên độ liên quan với câu hỏi"""
    def __init__(self, model_name=None):
        self.model_name = model_name
        # Sẽ triển khai trong tương lai
        pass
    
    def rerank(self, query, documents, top_k=5):
        """Xếp hạng lại các tài liệu dựa trên độ liên quan với câu truy vấn"""
        # Hiện tại trả về không thay đổi
        # TODO: Triển khai reranking ở đây
        return documents[:top_k]

# Khởi tạo reranker
reranker = Reranker()

In [ ]:
# Tạo Enhanced RAG Retriever
class EnhancedRetriever(BaseRetriever):
    """Trình truy xuất nâng cao kết hợp query expansion và reranking"""
    
    def __init__(self, base_retriever, query_expander=None, reranker=None, use_query_expansion=False, use_reranking=False):
        self.base_retriever = base_retriever
        self.query_expander = query_expander
        self.reranker = reranker
        self.use_query_expansion = use_query_expansion
        self.use_reranking = use_reranking
        
    def _get_relevant_documents(self, query, **kwargs):
        """Phương thức chính để truy xuất tài liệu"""
        if self.use_query_expansion and self.query_expander:
            # Mở rộng câu truy vấn
            expanded_queries = self.query_expander.expand_query(query)
            all_docs = []
            # Truy xuất với mỗi câu truy vấn mở rộng
            for expanded_query in expanded_queries:
                docs = self.base_retriever.get_relevant_documents(expanded_query)
                all_docs.extend(docs)
            # Loại bỏ trùng lặp
            unique_docs = []
            seen_ids = set()
            for doc in all_docs:
                if doc.metadata.get("source") not in seen_ids:
                    unique_docs.append(doc)
                    seen_ids.add(doc.metadata.get("source"))
        else:
            # Sử dụng retriever thông thường
            unique_docs = self.base_retriever.get_relevant_documents(query)
        
        # Áp dụng reranking nếu được bật
        if self.use_reranking and self.reranker:
            unique_docs = self.reranker.rerank(query, unique_docs)
            
        return unique_docs

# Tạo enhanced retriever với cấu trúc mở rộng trong tương lai
enhanced_retriever = EnhancedRetriever(
    base_retriever=retriever,
    query_expander=query_expander,
    reranker=reranker,
    use_query_expansion=False,  # Mặc định tắt cho đến khi triển khai
    use_reranking=False  # Mặc định tắt cho đến khi triển khai
)

In [ ]:
# Test retrieval with enhanced retriever
def test_retrieval(query, use_enhanced=False):
    print(f"\nQuery: {query}")
    
    if use_enhanced:
        # Sử dụng enhanced retriever
        docs = enhanced_retriever.get_relevant_documents(query)
        print(f"Retrieved {len(docs)} documents using EnhancedRetriever")
    else:
        # Sử dụng retriever cơ bản
        docs = retriever.get_relevant_documents(query)
        print(f"Retrieved {len(docs)} documents using standard Retriever")

    for i, doc in enumerate(docs):
        print(f"\nDocument {i+1}:")
        print(doc.page_content[:200] + "..." if len(doc.page_content) > 200 else doc.page_content)

# Test with a few queries
test_queries = [
    "Dịch bệnh nào dẫn đến nhiều cái chết giữa những năm 1830 và 1860?",
    "Ai đã thiết kế cánh mới cho cung điện vào năm 1847?"
]

for query in test_queries:
    test_retrieval(query)
    # Bỏ comment dòng dưới sau khi đã triển khai các phương pháp nâng cao
    # test_retrieval(query, use_enhanced=True)


Query: Dịch bệnh nào dẫn đến nhiều cái chết giữa những năm 1830 và 1860?
Retrieved 5 documents

Document 1:
Khoảng 1300 bóng1350 Thời kỳ ấm áp thời trung cổ đã nhường chỗ cho Kỷ băng hà nhỏ. Khí hậu lạnh hơn dẫn đến khủng hoảng nông nghiệp, lần đầu tiên được gọi là Nạn đói lớn 1315-1317. Tuy nhiên, hậu quả ...

Document 2:
Ở Anh, trong trường hợp không có số liệu thống kê dân số, các nhà sử học đề xuất một loạt các số liệu dân số trước từ 7 triệu đến thấp đến 4 triệu vào năm 1300, và dân số sau sinh là 2 triệu. Đến cuối...

Document 3:
Bệnh dịch hạch được giới thiệu lần đầu tiên tới châu Âu thông qua các thương nhân Genova tại thành phố cảng Kaffa ở Crimea vào năm 1347. Sau một cuộc bao vây kéo dài, trong đó quân đội Mông Cổ dưới qu...

Document 4:
Bệnh dịch liên tục quay trở lại ám ảnh châu Âu và Địa Trung Hải trong suốt thế kỷ 14 đến 17. Theo Biraben, bệnh dịch hạch đã xuất hiện ở đâu đó ở châu Âu vào mỗi năm từ 1346 đến 1671. Đại dịch thứ hai...

Document 5:
Đôi khi bị bỏ qua trong

In [ ]:
# Set up LLM
def setup_llm(model_name="vinai/phobert-base"):
    try:
        # Initialize tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # Add pad token if it doesn't exist
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Create HuggingFace pipeline
        pipe = pipeline(
            "text-generation",
            model=model_name,
            tokenizer=tokenizer,
            device=0 if torch.cuda.is_available() else -1,
            max_length=2024,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

        # Create LangChain LLM
        llm = HuggingFacePipeline(pipeline=pipe)
        return llm

    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None

# Initialize LLM
llm = setup_llm("meta-llama/Llama-3.2-1B-Instruct")
print("LLM initialized successfully!" if llm else "Failed to initialize LLM")

Device set to use cuda:0


LLM initialized successfully!


In [ ]:
# Create prompt template
template = """Dựa vào thông tin sau đây, hãy trả lời câu hỏi bằng tiếng Việt. Một câu hỏi tương ứng với mỗi câu trả lời.

Thông tin:
{context}

Câu hỏi: {question}

Trả lời:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# Create QA chain with option to use enhanced retriever
def create_qa_chain(use_enhanced_retriever=False):
    selected_retriever = enhanced_retriever if use_enhanced_retriever else retriever
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=selected_retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True
    )

# Create standard QA chain
qa_chain = create_qa_chain(use_enhanced_retriever=False)

In [ ]:
# Function to query the RAG system
def query_rag(question, use_enhanced_retriever=False):
    if not llm:
        # Fallback to retrieval only if LLM is not available
        selected_retriever = enhanced_retriever if use_enhanced_retriever else retriever
        docs = selected_retriever.get_relevant_documents(question)
        return {
            "query": question,
            "result": "LLM not available. Showing retrieved documents only.",
            "source_documents": docs
        }

    try:
        # Sử dụng QA chain phù hợp
        if use_enhanced_retriever:
            # Tạo QA chain với enhanced retriever nếu cần
            temp_qa_chain = create_qa_chain(use_enhanced_retriever=True)
            result = temp_qa_chain({"query": question})
        else:
            # Sử dụng QA chain tiêu chuẩn
            result = qa_chain({"query": question})
        return result
    except Exception as e:
        print(f"Error during RAG query: {e}")
        # Fallback to retrieval only
        selected_retriever = enhanced_retriever if use_enhanced_retriever else retriever
        docs = selected_retriever.get_relevant_documents(question)
        return {
            "query": question,
            "result": f"Error: {str(e)}",
            "source_documents": docs
        }

In [ ]:
# Test the RAG system
def display_rag_results(result):
    print(f"\nQuestion: {result['query']}")
    ans = result.get('result', 'No answer found.')

    # Extract the content after "Trả lời:" if it exists
    if "Trả lời:" in ans:
        ans = ans.split("Trả lời:", 1)[1].strip() # Split once and take the second part, then strip whitespace

    print(f"\nAnswer: {ans}")

# Run tests
for question in test_queries:
    result = query_rag(question)
    display_rag_results(result)
    # Bỏ comment dòng dưới khi đã triển khai các phương pháp nâng cao
    # result_enhanced = query_rag(question, use_enhanced_retriever=True)
    # print("\n--- Enhanced RAG Results ---")
    # display_rag_results(result_enhanced)


Question: Dịch bệnh nào dẫn đến nhiều cái chết giữa những năm 1830 và 1860?

Answer: Cái chết đen.

Question: Ai đã thiết kế cánh mới cho cung điện vào năm 1847?

Answer: Edward Blore.


In [ ]:
# Save and load functionality
def save_rag_system():
    # Save the vector store
    vectorstore.save_local("faiss_index")
    print("RAG system components saved successfully!")

def load_rag_system():
    # Load the vector store if it exists
    if os.path.exists("faiss_index"):
        loaded_vectorstore = FAISS.load_local("faiss_index", embeddings)
        loaded_retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 5})

        # Create enhanced retriever
        loaded_enhanced_retriever = EnhancedRetriever(
            base_retriever=loaded_retriever,
            query_expander=QueryExpansion(),
            reranker=Reranker(),
            use_query_expansion=False,
            use_reranking=False
        )

        # Recreate QA chain
        loaded_qa_chain = RetrievalQA.from_chain_type(
            llm=setup_llm(),
            chain_type="stuff",
            retriever=loaded_retriever,
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=True
        )

        print("RAG system loaded successfully!")
        return loaded_qa_chain, loaded_retriever, loaded_enhanced_retriever
    else:
        print("No saved RAG system found.")
        return None, None, None

save_rag_system()

RAG system components saved successfully!


# So sánh hiệu suất RAG với và không có Query Expansion/Reranking

Phần này sẽ so sánh hiệu quả của các phương pháp khác nhau khi được triển khai đầy đủ.

In [ ]:
# Hàm đánh giá và so sánh hiệu suất
def evaluate_rag_performance(test_questions, ground_truth_answers=None):
    """Đánh giá hiệu suất RAG với nhiều cấu hình khác nhau"""